In [1]:
"""
03_Embedding.py — Semantic Representation & Product Profiling
=============================================================
DiscoverAI · Health & Personal Care

This script builds the embedding layer for the semantic search system.
It produces three sets of vectors per product:
  1. Product embedding   — from product metadata (title + brand + description)
  2. Review embedding     — weighted average of individual review embeddings
  3. Combined embedding   — blended representation used for retrieval

Model : all-MiniLM-L6-v2  (384-dim, runs on CPU in ~5 min for ~7k products)
Input : data/products_cleaned.csv, data/reviews_topN.csv
Output: data/product_embeddings.npy, data/review_embeddings.npy,
        data/combined_embeddings.npy, data/embedding_index.csv
"""

# ── Imports ────────────────────────────────────────────────────────────────
import os
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_NAME      = "all-MiniLM-L6-v2"   # 384 dim, fast on CPU
BATCH_SIZE      = 64
MIN_TOKENS      = 10                    # from EDA decision
ALPHA           = 0.5                   # weight: product vs review embedding
DATA_DIR        = "Mean-Squared-Terrors/data"
SEED            = 42

# ── Paths ──────────────────────────────────────────────────────────────────
PRODUCTS_PATH   = os.path.join(DATA_DIR, "products_cleaned.csv")
REVIEWS_PATH    = os.path.join(DATA_DIR, "reviews_topN.csv")
OUT_PROD_EMB    = os.path.join(DATA_DIR, "product_embeddings.npy")
OUT_REV_EMB     = os.path.join(DATA_DIR, "review_embeddings.npy")
OUT_COMB_EMB    = os.path.join(DATA_DIR, "combined_embeddings.npy")
OUT_INDEX       = os.path.join(DATA_DIR, "embedding_index.csv")


def load_data():
    """Load products and reviews, apply the min-token filter from the EDA."""
    print("Loading data...")
    products = pd.read_csv(PRODUCTS_PATH)
    reviews  = pd.read_csv(REVIEWS_PATH)

    # token-length filter (decision from EDA notebook)
    reviews["tok_len"] = reviews["text"].astype(str).str.split().str.len()
    before = len(reviews)
    reviews = reviews[reviews["tok_len"] >= MIN_TOKENS].copy()
    print(f"  Reviews after tok_len >= {MIN_TOKENS}: {len(reviews):,} / {before:,} "
          f"({len(reviews)/before:.1%} kept)")
    print(f"  Products: {len(products):,}")
    return products, reviews


def embed_products(model, products):
    """Encode product_text_base for every product."""
    print("\n[1/3] Embedding products...")
    texts = products["product_text_base"].fillna("").tolist()
    t0 = time.time()
    embs = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,   # unit-norm → cosine sim = dot product
    )
    elapsed = time.time() - t0
    print(f"  Done — {embs.shape[0]} vectors × {embs.shape[1]} dims in {elapsed:.1f}s")
    return embs


def embed_reviews_aggregated(model, products, reviews):
    """
    For each product, embed its top-N reviews and aggregate via
    weighted average (weight = helpful_vote + 1).
    Returns one vector per product, aligned with the products dataframe.
    """
    print("\n[2/3] Embedding reviews and aggregating per product...")
    dim = model.get_sentence_embedding_dimension()
    asin_list = products["parent_asin"].tolist()

    # pre-group reviews by parent_asin for speed
    grouped = reviews.groupby("parent_asin")

    agg_embs = np.zeros((len(asin_list), dim), dtype=np.float32)
    products_without_reviews = 0

    t0 = time.time()
    for i, asin in enumerate(asin_list):
        if (i + 1) % 500 == 0:
            print(f"  Processing product {i+1}/{len(asin_list)}...")

        if asin not in grouped.groups:
            products_without_reviews += 1
            continue

        prod_revs = grouped.get_group(asin)
        texts   = prod_revs["text"].astype(str).tolist()
        weights = prod_revs["helpful_vote"].values.astype(float) + 1.0  # +1 to avoid zero-weight

        # encode this product's reviews
        embs = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

        # weighted average
        w = weights / weights.sum()
        agg = (embs * w[:, np.newaxis]).sum(axis=0)

        # re-normalize to unit length
        norm = np.linalg.norm(agg)
        if norm > 0:
            agg = agg / norm

        agg_embs[i] = agg

    elapsed = time.time() - t0
    print(f"  Done — {len(asin_list)} product-review vectors in {elapsed:.1f}s")
    if products_without_reviews:
        print(f"  ⚠ {products_without_reviews} products had no reviews after filtering "
              f"(zero vector fallback)")
    return agg_embs


def combine_embeddings(product_embs, review_embs, alpha=ALPHA):
    """Blend product and review embeddings and re-normalize."""
    print(f"\n[3/3] Combining embeddings (alpha={alpha})...")
    combined = alpha * product_embs + (1 - alpha) * review_embs
    norms = np.linalg.norm(combined, axis=1, keepdims=True)
    norms[norms == 0] = 1.0  # safety for zero vectors
    combined = combined / norms
    print(f"  Shape: {combined.shape}")
    return combined


def save_outputs(products, product_embs, review_embs, combined_embs):
    """Save all embeddings as .npy and the index mapping as CSV."""
    print("\nSaving outputs...")
    np.save(OUT_PROD_EMB, product_embs)
    np.save(OUT_REV_EMB,  review_embs)
    np.save(OUT_COMB_EMB, combined_embs)

    # index CSV: maps row position → parent_asin + product_title
    index_df = products[["parent_asin", "product_title"]].reset_index(drop=True)
    index_df.to_csv(OUT_INDEX, index=False)

    print(f"  ✓ {OUT_PROD_EMB}  ({product_embs.shape})")
    print(f"  ✓ {OUT_REV_EMB}   ({review_embs.shape})")
    print(f"  ✓ {OUT_COMB_EMB}  ({combined_embs.shape})")
    print(f"  ✓ {OUT_INDEX}     ({len(index_df)} rows)")


def quick_sanity_check(model, combined_embs, products):
    """Run a few test queries to verify the embeddings make sense."""
    print("\n" + "=" * 60)
    print("SANITY CHECK — test queries")
    print("=" * 60)

    test_queries = [
        "affordable moisturizer for sensitive skin",
        "electric toothbrush with long battery life",
        "vitamins for energy and immune support",
    ]

    for query in test_queries:
        q_emb  = model.encode([query], normalize_embeddings=True)
        scores = combined_embs @ q_emb.T
        top_5  = scores.flatten().argsort()[::-1][:5]

        print(f"\nQuery: \"{query}\"")
        for rank, idx in enumerate(top_5, 1):
            title = products.iloc[idx]["product_title"]
            score = scores[idx, 0]
            # truncate long titles
            if isinstance(title, str) and len(title) > 80:
                title = title[:77] + "..."
            print(f"  {rank}. [{score:.4f}] {title}")


# ── Main ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    np.random.seed(SEED)

    products, reviews = load_data()

    print(f"\nLoading model: {MODEL_NAME}")
    model = SentenceTransformer(MODEL_NAME)
    print(f"  Embedding dim: {model.get_sentence_embedding_dimension()}")

    product_embs = embed_products(model, products)
    review_embs  = embed_reviews_aggregated(model, products, reviews)
    combined_embs = combine_embeddings(product_embs, review_embs)

    save_outputs(products, product_embs, review_embs, combined_embs)

    quick_sanity_check(model, combined_embs, products)

    print("\n✅ Embedding complete. Ready for semantic search.")



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...
  Reviews after tok_len >= 10: 54,354 / 69,710 (78.0% kept)
  Products: 6,971

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21525.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dim: 384

[1/3] Embedding products...


Batches: 100%|██████████| 109/109 [00:17<00:00,  6.21it/s]


  Done — 6971 vectors × 384 dims in 17.6s

[2/3] Embedding reviews and aggregating per product...
  Processing product 500/6971...
  Processing product 1000/6971...
  Processing product 1500/6971...
  Processing product 2000/6971...
  Processing product 2500/6971...
  Processing product 3000/6971...
  Processing product 3500/6971...
  Processing product 4000/6971...
  Processing product 4500/6971...
  Processing product 5000/6971...
  Processing product 5500/6971...
  Processing product 6000/6971...
  Processing product 6500/6971...
  Done — 6971 product-review vectors in 429.5s
  ⚠ 1 products had no reviews after filtering (zero vector fallback)

[3/3] Combining embeddings (alpha=0.5)...
  Shape: (6971, 384)

Saving outputs...
  ✓ Mean-Squared-Terrors/data/product_embeddings.npy  ((6971, 384))
  ✓ Mean-Squared-Terrors/data/review_embeddings.npy   ((6971, 384))
  ✓ Mean-Squared-Terrors/data/combined_embeddings.npy  ((6971, 384))
  ✓ Mean-Squared-Terrors/data/embedding_index.csv     (69